In [1]:
import pandas as pd

df = pd.read_csv('/content/sensor_readings_24.csv')
print(df.head())

   0.438  0.498  3.625  3.645  5.000  2.918  5.000.1  2.351  2.332  2.643  \
0  0.438  0.498  3.625  3.648    5.0  2.918      5.0  2.637  2.332  2.649   
1  0.438  0.498  3.625  3.629    5.0  2.918      5.0  2.637  2.334  2.643   
2  0.437  0.501  3.625  3.626    5.0  2.918      5.0  2.353  2.334  2.642   
3  0.438  0.498  3.626  3.629    5.0  2.918      5.0  2.640  2.334  2.639   
4  0.439  0.498  3.626  3.629    5.0  2.918      5.0  2.633  2.334  2.645   

   ...  0.593  0.502  0.493  0.504  0.445  0.431  0.444  0.440  0.429  \
0  ...  0.592  0.502  0.493  0.504  0.449  0.431  0.444  0.443  0.429   
1  ...  0.593  0.502  0.493  0.504  0.449  0.431  0.444  0.446  0.429   
2  ...  0.593  0.502  0.493  0.504  0.449  0.431  0.444  0.444  0.429   
3  ...  0.592  0.502  0.493  0.504  0.449  0.431  0.444  0.441  0.429   
4  ...  0.589  0.502  0.493  0.504  0.446  0.431  0.444  0.444  0.430   

   Slight-Right-Turn  
0  Slight-Right-Turn  
1  Slight-Right-Turn  
2  Slight-Right-Turn  
3  Sli

In [2]:
print(df.isnull().sum())

0.438                0
0.498                0
3.625                0
3.645                0
5.000                0
2.918                0
5.000.1              0
2.351                0
2.332                0
2.643                0
1.698                0
1.687                0
1.698.1              0
1.717                0
1.744                0
0.593                0
0.502                0
0.493                0
0.504                0
0.445                0
0.431                0
0.444                0
0.440                0
0.429                0
Slight-Right-Turn    0
dtype: int64


In [3]:
non_numeric_cols = df.select_dtypes(exclude=['number'])
if not non_numeric_cols.empty:
    print("Non-numerical columns found:")
    for col in non_numeric_cols.columns:
        print(f"- {col}: {df[col].dtype}")
else:
    print("No non-numerical columns found.")

Non-numerical columns found:
- Slight-Right-Turn: object


In [4]:
from sklearn.preprocessing import LabelEncoder

# Initialize LabelEncoder
le = LabelEncoder()

# Encode the 'Slight-Right-Turn' column
df['Slight-Right-Turn_encoded'] = le.fit_transform(df['Slight-Right-Turn'])

# Display the mapping of original labels to encoded numbers
print("Original labels and their encoded values:")
for i, label in enumerate(le.classes_):
    print(f"- {label}: {i}")

# Drop the original non-numerical column if you no longer need it
df = df.drop('Slight-Right-Turn', axis=1)

print("\nDataFrame head after encoding:")
print(df.head())

Original labels and their encoded values:
- Move-Forward: 0
- Sharp-Right-Turn: 1
- Slight-Left-Turn: 2
- Slight-Right-Turn: 3

DataFrame head after encoding:
   0.438  0.498  3.625  3.645  5.000  2.918  5.000.1  2.351  2.332  2.643  \
0  0.438  0.498  3.625  3.648    5.0  2.918      5.0  2.637  2.332  2.649   
1  0.438  0.498  3.625  3.629    5.0  2.918      5.0  2.637  2.334  2.643   
2  0.437  0.501  3.625  3.626    5.0  2.918      5.0  2.353  2.334  2.642   
3  0.438  0.498  3.626  3.629    5.0  2.918      5.0  2.640  2.334  2.639   
4  0.439  0.498  3.626  3.629    5.0  2.918      5.0  2.633  2.334  2.645   

   ...  0.593  0.502  0.493  0.504  0.445  0.431  0.444  0.440  0.429  \
0  ...  0.592  0.502  0.493  0.504  0.449  0.431  0.444  0.443  0.429   
1  ...  0.593  0.502  0.493  0.504  0.449  0.431  0.444  0.446  0.429   
2  ...  0.593  0.502  0.493  0.504  0.449  0.431  0.444  0.444  0.429   
3  ...  0.592  0.502  0.493  0.504  0.449  0.431  0.444  0.441  0.429   
4  ...  0.589

In [5]:
import numpy as np

# Assuming 'Slight-Right-Turn_encoded' is the target column and should not be considered for outlier detection
feature_cols = [col for col in df.columns if col != 'Slight-Right-Turn_encoded']

outlier_counts = {}

for col in feature_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    outlier_counts[col] = len(outliers)

print("Number of outliers per feature column (using IQR method):")
for col, count in outlier_counts.items():
    print(f"- {col}: {count} outliers")

# Optionally, you can also calculate the total number of outliers across all feature columns
total_outliers = sum(outlier_counts.values())
print(f"\nTotal number of outliers across all feature columns: {total_outliers}")

Number of outliers per feature column (using IQR method):
- 0.438: 197 outliers
- 0.498: 966 outliers
- 3.625: 873 outliers
- 3.645: 0 outliers
- 5.000: 0 outliers
- 2.918: 0 outliers
- 5.000.1: 0 outliers
- 2.351: 0 outliers
- 2.332: 0 outliers
- 2.643: 0 outliers
- 1.698: 0 outliers
- 1.687: 828 outliers
- 1.698.1: 916 outliers
- 1.717: 0 outliers
- 1.744: 0 outliers
- 0.593: 502 outliers
- 0.502: 522 outliers
- 0.493: 559 outliers
- 0.504: 726 outliers
- 0.445: 757 outliers
- 0.431: 480 outliers
- 0.444: 1000 outliers
- 0.440: 718 outliers
- 0.429: 548 outliers

Total number of outliers across all feature columns: 9592


In [6]:
for col in feature_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Cap the outliers
    df[col] = np.where(df[col] < lower_bound, lower_bound, df[col])
    df[col] = np.where(df[col] > upper_bound, upper_bound, df[col])

print("Outliers have been capped in the DataFrame. Displaying head to confirm:")
print(df.head())

Outliers have been capped in the DataFrame. Displaying head to confirm:
   0.438  0.498  3.625  3.645  5.000  2.918  5.000.1  2.351  2.332  2.643  \
0  0.438  0.498  3.625  3.648    5.0  2.918      5.0  2.637  2.332  2.649   
1  0.438  0.498  3.625  3.629    5.0  2.918      5.0  2.637  2.334  2.643   
2  0.437  0.501  3.625  3.626    5.0  2.918      5.0  2.353  2.334  2.642   
3  0.438  0.498  3.626  3.629    5.0  2.918      5.0  2.640  2.334  2.639   
4  0.439  0.498  3.626  3.629    5.0  2.918      5.0  2.633  2.334  2.645   

   ...  0.593  0.502  0.493  0.504  0.445  0.431  0.444  0.440  0.429  \
0  ...  0.592  0.502  0.493  0.504  0.449  0.431  0.444  0.443  0.429   
1  ...  0.593  0.502  0.493  0.504  0.449  0.431  0.444  0.446  0.429   
2  ...  0.593  0.502  0.493  0.504  0.449  0.431  0.444  0.444  0.429   
3  ...  0.592  0.502  0.493  0.504  0.449  0.431  0.444  0.441  0.429   
4  ...  0.589  0.502  0.493  0.504  0.446  0.431  0.444  0.444  0.430   

   Slight-Right-Turn_encod

In [7]:
from sklearn.model_selection import train_test_split

# Define features (X) and target (y)
X = df[feature_cols]
y = df['Slight-Right-Turn_encoded']

# Split the data into training and testing sets (e.g., 80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (4364, 24)
X_test shape: (1091, 24)
y_train shape: (4364,)
y_test shape: (1091,)


In [8]:
print("Columns in X (features DataFrame) are:")
print(X.columns.tolist())

Columns in X (features DataFrame) are:
['0.438', '0.498', '3.625', '3.645', '5.000', '2.918', '5.000.1', '2.351', '2.332', '2.643', '1.698', '1.687', '1.698.1', '1.717', '1.744', '0.593', '0.502', '0.493', '0.504', '0.445', '0.431', '0.444', '0.440', '0.429']


In [9]:
from sklearn.preprocessing import MinMaxScaler

# Initialize MinMaxScaler
scaler = MinMaxScaler()

# Apply Min-Max scaling to X_train
X_train_scaled = scaler.fit_transform(X_train)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)

# Apply the *same* scaling to X_test (do not refit)
X_test_scaled = scaler.transform(X_test)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print("X_train_scaled head:")
print(X_train_scaled.head())
print("\nX_test_scaled head:")
print(X_test_scaled.head())

X_train_scaled head:
      0.438     0.498     3.625     3.645     5.000     2.918   5.000.1  \
0  0.426730  0.272501  0.280607  0.995937  1.000000  0.998715  0.154222   
1  0.195751  0.252158  0.253824  0.164914  0.088918  0.998715  0.998455   
2  0.336662  1.000000  0.906444  0.793977  1.000000  0.462863  0.934861   
3  0.364264  0.225192  0.223601  0.132648  0.075773  0.083269  0.386457   
4  0.518613  0.334477  0.345476  0.995937  0.220103  0.216397  0.219104   

      2.351     2.332     2.643  ...     1.744     0.593     0.502     0.493  \
0  0.201750  0.212776  0.327924  ...  0.164484  0.587724  0.886320  0.257552   
1  0.745506  1.000000  1.000000  ...  0.071920  0.164145  0.265896  0.320085   
2  0.357616  0.190922  0.202625  ...  0.077026  0.447922  0.363198  0.255432   
3  0.564333  0.580692  1.000000  ...  0.080133  0.306034  0.500963  0.754637   
4  0.271523  0.332373  0.319093  ...  0.314983  0.835333  1.000000  1.000000   

      0.504     0.445     0.431     0.444     0

In [10]:
df.head()

,0.438,0.498,3.625,3.645,5.000,2.918,5.000.1,2.351,2.332,2.643,...,0.593,0.502,0.493,0.504,0.445,0.431,0.444,0.440,0.429,Slight-Right-Turn_encoded
0,0.438,0.498,3.625,3.648,5.0,2.918,5.0,2.637,2.332,2.649,...,0.592,0.502,0.493,0.504,0.449,0.431,0.444,0.443,0.429,3
1,0.438,0.498,3.625,3.629,5.0,2.918,5.0,2.637,2.334,2.643,...,0.593,0.502,0.493,0.504,0.449,0.431,0.444,0.446,0.429,3
2,0.437,0.501,3.625,3.626,5.0,2.918,5.0,2.353,2.334,2.642,...,0.593,0.502,0.493,0.504,0.449,0.431,0.444,0.444,0.429,3
3,0.438,0.498,3.626,3.629,5.0,2.918,5.0,2.640,2.334,2.639,...,0.592,0.502,0.493,0.504,0.449,0.431,0.444,0.441,0.429,3
4,0.439,0.498,3.626,3.629,5.0,2.918,5.0,2.633,2.334,2.645,...,0.589,0.502,0.493,0.504,0.446,0.431,0.444,0.444,0.430,3


In [13]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical
import numpy as np
from sklearn.metrics import classification_report

# Assuming y_train and y_test are already defined as integer labels
# Check the number of unique classes in the target variable
num_classes = len(np.unique(y_train))

# One-hot encode the target variables
y_train_one_hot = to_categorical(y_train, num_classes=num_classes)
y_test_one_hot = to_categorical(y_test, num_classes=num_classes)

# Get the number of features from the scaled training data
input_shape = X_train_scaled.shape[1]

# Design the Keras MLP model with two hidden layers
model = Sequential([
    Dense(128, activation='relu', input_shape=(input_shape,)), # First hidden layer
    Dense(64, activation='relu'),                               # Second hidden layer
    Dense(num_classes, activation='softmax')                    # Output layer for multi-class classification
])

# Compile the model
# Using 'adam' optimizer, 'categorical_crossentropy' for one-hot encoded targets,
# and 'accuracy' as a metric.
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Print a summary of the model architecture
print("Keras MLP Model Summary:")
model.summary()

print(f"\nStarting training for {100} epochs...")

# Train the model
history = model.fit(X_train_scaled, y_train_one_hot, epochs=100, batch_size=32, validation_split=0.1, verbose=0) # verbose=0 to suppress per-epoch output

print("\nTraining complete. Evaluating the model...")

# Evaluate the model on the test data
loss, accuracy = model.evaluate(X_test_scaled, y_test_one_hot, verbose=0)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

# Find the best epoch based on validation accuracy
best_epoch = np.argmax(history.history['val_accuracy']) + 1 # +1 because epochs are 1-indexed
best_val_accuracy = history.history['val_accuracy'][best_epoch - 1]
best_val_loss = history.history['val_loss'][best_epoch - 1]
print(f"\nBest epoch (based on validation accuracy): {best_epoch}")
print(f"Best Validation Accuracy at epoch {best_epoch}: {best_val_accuracy:.4f}")
print(f"Validation Loss at best epoch {best_epoch}: {best_val_loss:.4f}")

# Generate predictions for classification report
y_train_pred_probs = model.predict(X_train_scaled)
y_test_pred_probs = model.predict(X_test_scaled)

# Convert probabilities to class labels
y_train_pred = np.argmax(y_train_pred_probs, axis=1)
y_test_pred = np.argmax(y_test_pred_probs, axis=1)

# Get true labels from one-hot encoded data (if needed, otherwise use original y_train, y_test)
# y_train_true = np.argmax(y_train_one_hot, axis=1)
# y_test_true = np.argmax(y_test_one_hot, axis=1)

print("\n--- Classification Report (Training Set) ---")
print(classification_report(y_train, y_train_pred))

print("\n--- Classification Report (Test Set) ---")
print(classification_report(y_test, y_test_pred))

print("\nKeras MLP model has been trained and evaluated.")


Keras MLP Model Summary:


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 128)            │         3,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,716 (45.77 KB)

 Trainable params: 11,716 (45.77 KB)

 Non-trainable params: 0 (0.00 B)


Starting training for 100 epochs...

Training complete. Evaluating the model...
Test Loss: 0.1890
Test Accuracy: 0.9468

Best epoch (based on validation accuracy): 57
Best Validation Accuracy at epoch 57: 0.9657
Validation Loss at best epoch 57: 0.1655
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 

--- Classification Report (Training Set) ---
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1773
           1       0.99      0.99      0.99      1662
           2       1.00      1.00      1.00       263
           3       0.99      0.98      0.98       666

    accuracy                           0.99      4364
   macro avg       0.99      0.99      0.99      4364
weighted avg       0.99      0.99      0.99      4364


--- Classification Report (Test Set) ---
              precision    recall  f1-score   support

           0       0.95      0.94      0.95       432
           1       0.96      0.95